# Annotations — building the combined annotation table

Stage 2 of three. Produces **`output/annotated_combined.tsv`**, the single table
every figure notebook reads.

```
Scoring.ipynb          raw barcode counts  ->  scores + identity   (stage 1)
  score_cell.py / finalize_scores.py           38 columns, no annotations
        |
        v
reannotate_scores.py   primary annotation files  ->  scores_reannotated.tsv
  + compute_structure_annotations.py            92 columns          (stage 2a)
        |
        v
THIS NOTEBOOK          adds the project-specific layers              (stage 2b)
        |
        v
DN.ipynb / HSP90i.ipynb                        figures              (stage 3)
```

Every annotation is rebuilt from a primary file — the AlphaFold models, the DSSP
outputs, the NCBI conserved-domain tables, PhosphoSitePlus, ClinVar,
AlphaMissense, CysDB, the published DMS tables, the kinase alignment, and the
curated domain bounds in `config/proteins.yaml`.
`docs/scores_reannotated_columns.md` documents every column of the stage-2a table; `docs/raw_scores_columns.md` the smaller raw table.

**Inputs**
| File | Role |
|---|---|
| `output/scoring/scores_reannotated.tsv` | scores plus the rebuilt annotation layer |
| `data/dn_cutoffs_empty_vector.tsv` | per-library empty-vector DN thresholds |
| `output/dn/interactions_curated_augmented.tsv` | curated PDB interfaces + 4 paralog/structure augmentations |
| `output/hsp90/transferred_contacts.tsv` | HSP90/CDC37 contacts, 4 Å all-atom, split unfolded/folded |
| `output/population_data_hgvsp.tsv` | gnomAD / All of Us observation flags |
| `output/clinical/activity_vs_genie_joined.tsv` | GENIE tumour counts |
| `data/inputs/spurs_ddg_all_proteins.tsv`, `output/dn/phylop_per_position.tsv`, `output/hsp90/kinase_jsd_per_position.tsv` | quantitative predictors |

**What stage 2a provides** — structure (pLDDT, the model's own residue identity,
DSSP secondary structure and solvent accessibility, relative SASA, 4 Å all-atom
inter-domain contacts), curated domains, NCBI conserved-domain features and active
sites, PPI interfaces, PTMs and regulatory sites from PhosphoSitePlus, ClinVar,
AlphaMissense pathogenicity, CysDB ligandable cysteines, OpenCell expression, the
published SHP2 DMS comparison, kinase-alignment positions, z-scores against the
synonymous-WT distribution, and the chaperone-dependency and buffering families.

**What this notebook adds** — `DN_EV` (empty-vector dominant negatives), the
curated-PDB interface partners, our HSP90/CDC37 contacts split at the unfolded
N-lobe boundary, phyloP and kinase-JSD conservation, SPURS ΔΔG, kinase motifs,
population and GENIE observation, and the verified reference accessions.

**What is not used downstream.** `Abund_chaperone_dependency` and
`chap_dep_classification` are named *dependence* but are computed on
`average score`, which re-anchors wild type to 1.0 in each condition and so
cancels the very shift dependence is about. `buffered` and `percent_buffered` are
named *buffering* but are on the absolute scale, and
`percent_buffered_WT_norm` divides by a near-zero wild-type value for the
non-clients. The buffering classes the figures use are assigned in
`HSP90i.ipynb`, not here — see `docs/hsp90_metric_definitions.md`.

**Known upstream issues, tracked rather than silently absorbed** — logged in the
provenance report at the end.


In [1]:
%load_ext autoreload
%autoreload 2
# Edits to utils.py take effect without restarting the kernel. Note that
# autoreload re-executes changed module bodies, so any module-level constant
# reassigned from the notebook (e.g. utils.BUFFERING_COLORS = ...) is reset on
# reload — pass such overrides as function arguments instead.

In [2]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import utils
# Every input this notebook reads is named here, via utils, rather than being
# built from a string further down. utils.input_manifest() then checks each one
# exists before any of them is opened -- a missing or truncated input becomes an
# error at the top instead of a puzzling number in a figure.
from utils import (
    SCORES_REANNOTATED,        # stage 2a: our scores, annotated from primary files
    ANNOTATED_COMBINED,        # what this notebook writes; the figures' input
    STRUCTURE_ANNOTATIONS,     # AlphaFold pLDDT / residue identity / inter-domain
    DN_THRESHOLDS,             # DN thresholds recomputed from EV barcodes
    CONTROL_BARCODE_SCORES,    # the EV / NoVar barcode scores behind them
    INTERACTIONS_AUGMENTED,    # curated PDB interfaces + 4 augmentations
    TRANSFERRED_CONTACTS,      # HSP90/CDC37 contacts, 4 A, split by lobe
    KINASE_MOTIFS,             # canonical kinase motif per alignment column
    PHYLOP_PER_POSITION,       # cross-species conservation
    KINASE_JSD,                # within-kinome (paralog) conservation
    SPURS_DDG,                 # predicted folding ddG
    POPULATION_HGVSP,          # gnomAD / All of Us observation
    GENIE_JOINED,              # GENIE tumour counts
    REFERENCE_FASTA,           # for the wild-type sequence audit
    PROTEIN_ACCESSIONS,        # verified UniProt / Ensembl / RefSeq accessions
    OUTPUT, PROJECT_ROOT,
    SCORE_WT_REL, SCORE_ABS, SCORE_ABS_REPS,
)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)
warnings.filterwarnings("ignore", category=FutureWarning)

# Aliases kept so the cells below read the way they always have.
CONTACTS, MOTIFS, PHYLOP = TRANSFERRED_CONTACTS, KINASE_MOTIFS, PHYLOP_PER_POSITION

manifest = utils.input_manifest()
_missing = manifest[~manifest.exists]
print(f"inputs: {len(manifest)}   missing: {len(_missing)}")
if len(_missing):
    display(_missing[["input", "path"]])
    raise FileNotFoundError(
        f"{len(_missing)} input file(s) missing — see the table above")
display(manifest[["input", "size"]].head(15))
print("... and", len(manifest) - 15, "more (annotation engine sources); "
      "all present")

#: Collected as we go and printed at the end, so nothing is absorbed silently.
provenance: list = []


inputs: 37   missing: 0


,input,size
0,scores (stage 1),279.9 MB
1,scores re-annotated (stage 2a),474.2 MB
2,structure annotations,0.5 MB
3,AlphaFold models,19 files
4,control barcode scores,14.1 MB
5,DN thresholds (recomputed),0.0 MB
6,curated interactions,0.0 MB
7,HSP90/CDC37 contacts,0.0 MB
8,kinase motifs,0.1 MB
9,phyloP,0.2 MB


... and 22 more (annotation engine sources); all present


## Step 1 — load the score table and audit the experimental grid

The audit is here so that a structural change in the inputs shows up as a diff
in this cell's output rather than as a puzzling number three notebooks later.
Two treatment labels differ from an earlier pipeline run: KRAS activity
`No_treatment` → `DMSO`, and KSR2 C-term abundance `DMSO` → `No_treatment`.
Both are deliberate relabels of the same experiments.

In [3]:
scores = pd.read_csv(SCORES_REANNOTATED, sep="\t", low_memory=False)
print(f"re-annotated table: {scores.shape[0]:,} rows x {scores.shape[1]} columns")
print(f"  {SCORES_REANNOTATED.relative_to(PROJECT_ROOT)}")

grid = (scores.groupby(["assay", "assay_treatment"]).size()
        .rename("rows").reset_index())
display(grid)

print("libraries:", scores.library.nunique(),
      "| proteins:", scores.protein.nunique())
display(scores["Mutation Type"].value_counts().rename("rows").to_frame())

provenance.append(
    f"Base layer is {SCORES_REANNOTATED.name} ({scores.shape[0]:,} rows): our "
    f"scores, built from our own barcode counts by Scoring.ipynb, with every "
    f"annotation rebuilt from primary sources by scripts/reannotate_scores.py "
    f"and scripts/compute_structure_annotations.py.")


re-annotated table: 531,797 rows x 96 columns
  output/scoring/scores_reannotated.tsv


,assay,assay_treatment,rows
0,abundance,CIAR,20928
1,abundance,DMSO,53069
2,abundance,HSP90i,111374
3,abundance,LZTR1ko,4238
4,abundance,LZTR1koCIAR,4238
5,abundance,No_treatment,122614
6,activity,CIAR,4336
7,activity,DMSO,4343
8,activity,No_treatment,192458
9,activity,SerumStarve,10372


libraries: 22 | proteins: 17


,rows
Mutation Type,
missense,430858
synonymous wild type,21755
deletion,20777
frame shift,20703
nonsense,17235
delins_2for1,12474
multi_change,7165
standard,755
wild type,64


## Step 3 — normalise columns

Two mechanical fixes and a set of renames that make the measurement scale part
of the column name.

`feature` is a Python list literal inside a TSV, so it is parsed to a real
list. A residue with no feature comes through as null (470,520 rows), not as the
string `['none']` — a placeholder that would make any `.notna()` check evaluate
to True everywhere. The check for it is kept so either shape works.

The renames are the substantive part. `protein_interface` (conserved-domain
feature tags) and the curated-PDB interface set are different quantities, so
both survive under names that say what they are; nothing is overwritten.
`client_status` becomes `client_status_literature`, because it is a literature
assignment that calls ERBB2 and KSR1 "strong" although neither was tested with
HSP90i here (G6).

In [4]:
import ast

ann = scores.copy()

# -- feature: parse the list literal ---------------------------------------
# The annotation engine emits `feature` as a Python list literal inside a TSV.
# Parse it to a readable string. A `['none']` placeholder does not occur -- the
# engine leaves a residue with no feature as NA -- but the check is kept so that
# either shape works.
def _parse_feature(v):
    if not isinstance(v, str):
        return np.nan
    try:
        parsed = ast.literal_eval(v)
    except (ValueError, SyntaxError):
        return np.nan
    if not isinstance(parsed, list) or parsed in ([], ["none"]):
        return np.nan
    return "; ".join(sorted(parsed))

_before = ann["feature"].notna().sum()
ann["feature"] = ann["feature"].map(_parse_feature)
print(f"feature non-null: {_before:,} -> {ann['feature'].notna().sum():,}")

# -- booleans are already booleans -----------------------------------------
# The annotation engine emits real booleans, so a string conversion here would
# be not merely unnecessary but actively wrong: `.astype(str).eq("yes")` on a bool
# column yields False everywhere and would silently erase the flag.
_BOOLS = ["active_site", "protein_interface",
          "inter_domain_contacts_all_atom", "pdb_aa_mismatch", "regulatory_PTM"]
for c in _BOOLS:
    assert ann[c].dtype == bool, (
        f"{c} is {ann[c].dtype}, expected bool — the re-annotation changed shape")
print(f"boolean flags verified as bool dtype: {_BOOLS}")

# -- renames: scale and provenance into the name ---------------------------
# `client_status` is a literature assignment, not measured here (G6): it calls
# ERBB2 and KSR1 "strong" though neither was tested with HSP90i in this study.
ann = ann.rename(columns={"client_status": "client_status_literature"})

# `protein_interface` is the curated-PDB contact set computed by
# add_ppi_interfaces (106,277 rows flagged), so it is renamed for what it is.
# The engine reads the *augmented* curated interactions file (config/paths.yaml),
# so `protein_interface` is already the full set: the curated PDB contacts plus
# the RET dimer from 7JU5, SOS2's catalytic surface from SOS1, the ARAF dimer
# from BRAF, and MEK2's seven surfaces from MEK1. Renamed for what it is. Step 5
# adds the partner names to the same positions; it no longer recomputes the flag.
ann = ann.rename(columns={"protein_interface": "interface_pdb_curated"})
provenance.append(
    "Interfaces: one column, `interface_pdb_curated`, from the augmented curated "
    "interactions file, which the annotation engine reads directly. Chaperone "
    "contacts are computed from 7ZR0 and 8U1L in Step 6.")


feature non-null: 61,277 -> 61,277
boolean flags verified as bool dtype: ['active_site', 'protein_interface', 'inter_domain_contacts_all_atom', 'pdb_aa_mismatch', 'regulatory_PTM']


## Step 4 — dominant negatives (`DN_EV`)

A variant is dominant negative when it drives pathway activity *below the level
seen with no variant protein at all*, which the per-library empty-vector
controls define directly. Four restrictions, each with a reason:

1. **Activity assay, baseline treatments only** (`No_treatment` / `DMSO`).
   Serum starvation collapses EGFR wild type onto the empty-vector baseline,
   leaving almost no headroom, so "below the floor" can no longer separate
   dominant negativity from ordinary loss of function. CIAR drives the pathway
   independently of KRAS variant identity, so that readout is not reporting
   variant function either.
2. **Protein-altering types only.** Synonymous variants and the BRAF spike-in
   standards cannot be dominant negative.
3. **Inhibitory proteins excluded** (GRB2, KSR1, KSR2, MEK1, MEK2). Their
   wild-type overexpression *lowers* pathway activity, so scoring below the
   empty vector is the expected wild-type phenotype rather than a defect.
4. **Thresholds keyed per (library, treatment)**, never pooled, because
   empty-vector activity spans orders of magnitude across libraries.

The cutoff file predates the KRAS relabel, so its `kras No_treatment` row is
remapped to `kras DMSO`. It is the same experiment under a new name.

In [5]:
# Thresholds recomputed from our own empty-vector barcodes in Scoring.ipynb
# Step 11 — a low percentile of resampled barcode means, which is the
# distribution a *variant* score is drawn from when the construct is empty.
#
# No B10 alias is needed any more: the thresholds are keyed by the treatment
# labels in our own data, so KRAS activity is `DMSO` on both sides by
# construction rather than by a remap.
ev = pd.read_csv(DN_THRESHOLDS, sep="\t")
# `used` is the control set Step 11 decided on per cell: empty vector pooled with
# NoVar_std in untreated cells, empty vector alone in treated ones, because the
# spiked standards are never treated while the empty vector piggybacks on the
# library and is. Pooling substantially enlarges the draw pool in the cells
# where empty vector alone is thinnest.
ev = ev[(ev.assay == "activity") & (ev.control == "used")]

# Only untreated cells are DN baselines. EGFR SerumStarve and KRAS CIAR carry a
# threshold in the table but are deliberately not DN baselines: serum starvation
# collapses EGFR-WT onto the empty-vector level, leaving no headroom to separate
# dominant negativity from ordinary loss of function, and CIAR drives the pathway
# independently of KRAS variant identity.
ev_baseline = ev[ev.assay_treatment.isin(utils.BASELINE_TREATMENTS)]
_excluded = sorted(set(map(tuple, ev.loc[~ev.assay_treatment.isin(
    utils.BASELINE_TREATMENTS), ["library", "assay_treatment"]].values)))
print(f"treated cells excluded from DN consideration: {_excluded}")
thr = ev_baseline.set_index(["library", "assay_treatment"])["dn_threshold"]
provenance.append(
    f"DN thresholds recomputed from {CONTROL_BARCODE_SCORES.name} rather than "
    f"read from the delivered dn_cutoffs_empty_vector.tsv: 2.5th percentile of "
    f"200 draws of a 10-barcode mean, median over 200 seed repeats. "
    f"{len(ev_baseline)} baseline activity cells carry a threshold.")

eligible = (
    (ann.assay == "activity")
    & ann.assay_treatment.isin(utils.BASELINE_TREATMENTS)
    & ann["Mutation Type"].isin(utils.PROTEIN_ALTERING_TYPES)
    & ~ann.protein.isin(utils.INHIBITORY_PROTEINS)
)
cut = pd.MultiIndex.from_arrays([ann.library, ann.assay_treatment]).map(thr)
ann["dn_threshold"] = np.where(eligible, cut, np.nan)

# object dtype: True / False / None, so "not assessable" never counts as
# "not DN" in a groupby mean.
dn = pd.Series(None, index=ann.index, dtype=object)
assessable = eligible & ann["dn_threshold"].notna() & ann[SCORE_WT_REL].notna()
dn[assessable] = (ann.loc[assessable, SCORE_WT_REL]
                  < ann.loc[assessable, "dn_threshold"]).values
ann["DN_EV"] = dn

n_dn = int((ann.DN_EV == True).sum())
print(f"DN_EV True: {n_dn:,} rows   assessable: {int(assessable.sum()):,}")
display(ann[ann.DN_EV == True].groupby(["protein", "Mutation Type"])
        .size().unstack(fill_value=0)
        .reindex([p for p in utils.DN_ELIGIBLE]).fillna(0).astype(int))

treated cells excluded from DN consideration: [('egfr', 'SerumStarve'), ('kras', 'CIAR')]


DN_EV True: 13,791 rows   assessable: 116,559


Mutation Type,deletion,missense,nonsense
protein,,,
araf,218,3017,399
braf,154,2685,364
craf,43,335,124
egfr,30,277,18
erbb2,0,30,11
kras,40,214,1
met,8,79,26
mras,49,586,10
ret,54,1540,1


## Step 5 — interfaces from curated PDB contacts

Two different quantities are available and only one is used. Conserved-domain
feature tags mark 214 positions across the nine HSP90i kinases, including
"putative" calls and ligand-binding pockets. Per-residue contact from curated
PDB structures, with the partner recorded, marks 989 positions over the same
proteins — 4.6× more — and for ARAF, MEK2, HGFR and RET the two sets do not
overlap at all. The contact-based set is what the figures use.

The source is `interactions_curated_augmented.tsv` — the curated interface file
plus four augmentations that fill gaps which are missing curation rather than
biology: the RET kinase dimer from 7JU5, SOS2's catalytic surface projected
from SOS1, the ARAF dimer projected from BRAF, and (added 2026-08-02) MEK2's
seven partner surfaces projected from MEK1 at 81.9% identity. The MEK1→MEK2
projection was verified against the documented catalytic-lysine
correspondence, MEK1 K97 → MEK2 K101.

Both flags are kept: `interface_cd_annotated` for the feature tags,
`interface_pdb_curated` for the contacts, which is what the figures use.

In [6]:
aug = pd.read_csv(INTERACTIONS_AUGMENTED, sep="\t", index_col=0)
aug = aug[aug["keep"].astype(bool)]

def _positions(s):
    if pd.isna(s):
        return []
    try:
        v = ast.literal_eval(str(s).strip().strip('"'))
    except (ValueError, SyntaxError):
        return []
    return [int(x) for x in v] if isinstance(v, list) else []

def _partner_name(row, side):
    other = "protein_a" if side == "protein_b" else "protein_b"
    if other == "protein_a":
        nm = str(row.get("protein_a_name", "") or "").strip()
        if nm and nm.lower() != "nan":
            return nm.lower()
    return str(row[other]).strip().lower()

# position -> set of partners, per protein
partners: dict[tuple[str, int], set[str]] = {}
for _, row in aug.iterrows():
    for side, pos_col in (("protein_a", "positions_a"),
                          ("protein_b", "positions_b")):
        prot = str(row[side]).strip().lower()
        if not prot or prot == "nan":
            continue
        who = _partner_name(row, side)
        for p in _positions(row[pos_col]):
            partners.setdefault((prot, p), set()).add(who)

pos_num = pd.to_numeric(ann["Position"], errors="coerce")
keys = list(zip(ann["protein"], pos_num.fillna(-1).astype(int)))

# The flag itself already exists: the engine reads this same augmented file, so
# `interface_pdb_curated` is set. What is added here is *who* each position
# contacts, which the engine does not record. Recomputing the flag would create
# a second copy of the same measurement, so instead it is asserted to agree --
# if the two ever diverge, the config and this cell are reading different files.
ann["interface_partners"] = ["; ".join(sorted(partners[k])) if k in partners
                             else np.nan for k in keys]
ann["n_interface_partners"] = (ann["interface_partners"].fillna("")
                               .map(lambda s: len(s.split("; ")) if s else 0))

_from_partners = ann["interface_partners"].notna()
_disagree = int((_from_partners != ann["interface_pdb_curated"]).sum())
assert _disagree == 0, (
    f"{_disagree:,} rows where the partner lookup and the engine's "
    "`interface_pdb_curated` disagree — config/paths.yaml `interactions` and "
    "INTERACTIONS_AUGMENTED are not the same file")
print(f"interface positions agree with the engine on all {len(ann):,} rows")

cmp = (ann[ann.interface_pdb_curated]
       .groupby("protein")
       .apply(lambda g: pd.to_numeric(g.Position, errors="coerce").nunique())
       .reindex(utils.PROTEIN_KEYS).fillna(0).astype(int)
       .rename("unique interface positions").to_frame())
cmp["partners"] = (ann[ann.interface_pdb_curated].groupby("protein")
                   ["interface_partners"].apply(
                       lambda s: len({p for v in s.dropna()
                                      for p in v.split("; ")}))
                   .reindex(utils.PROTEIN_KEYS).fillna(0).astype(int))
print("curated PDB interfaces (augmented file):")
display(cmp)
provenance.append(
    f"Interfaces: {int(cmp['unique interface positions'].sum()):,} unique "
    f"positions across {int((cmp['unique interface positions'] > 0).sum())} "
    "proteins, from the augmented curated file, with the contacting partner "
    "recorded per position. The augmentations are four projected surfaces (RET "
    "dimer from 7JU5, SOS2 from SOS1, ARAF from BRAF, MEK2 from MEK1); without "
    "them SOS2 would have 0 annotated interface positions and RET 2.")

interface positions agree with the engine on all 531,797 rows
curated PDB interfaces (augmented file):


,unique interface positions,partners
protein,,
araf,93,4
braf,203,8
craf,220,11
egfr,188,8
erbb2,19,5
met,15,3
ret,38,3
kras,100,17
mras,45,5


## Step 6 — HSP90 and CDC37 contacts

4 Å all-atom contacts from the BRAF (7ZR0, V600E) and CRAF (8U1L)
chaperone-complex structures, transferred to all 11 kinases through the
497-kinase alignment and **split at the unfolded-N-lobe boundary** into
HSP90-unfolded / HSP90-folded / CDC37. Fig 4's argument — that what confers
buffering is the state of the fold rather than whether the kinase can still be
recognised — depends on that split.

A third state, **`Unknown`**, is kept for residues no structure resolves. A bare
binary flag treats them as non-contacts, which contaminates the control group in
precisely the test that ranked chaperone contacts near the bottom of the ~50
features. The `*_resolved` columns below let that test be run with unresolved
positions excluded rather than counted as negatives.

In [7]:
ct = pd.read_csv(CONTACTS, sep="\t")
ct = ct[ct.category.isin(["HSP90_unf", "HSP90_fld", "CDC37"])]
sets = {cat: set(zip(g.protein, g.position.astype(int)))
        for cat, g in ct.groupby("category")}

for cat, col in (("HSP90_unf", "at_hsp90_unfolded"),
                 ("HSP90_fld", "at_hsp90_folded"),
                 ("CDC37", "at_cdc37")):
    ann[col] = [k in sets[cat] for k in keys]
ann["at_any_chaperone_contact"] = (ann.at_hsp90_unfolded | ann.at_hsp90_folded
                                   | ann.at_cdc37)

# "Was this residue in a position where a contact could have been seen at all?"
# An explicit third state for residues no structure resolves, which is better
# than a bare binary: it keeps unresolved positions out of the control group
# instead of counting them as non-contacts. Contacts reach a kinase through the
# 497-kinase alignment, so a
# residue is assessable exactly when it maps to an alignment column; one that
# does not is unresolved, not a negative.
ann["chaperone_contact_resolved"] = ann["alignment_pos"].notna()
provenance.append(
    "`chaperone_contact_resolved` is now derived from `alignment_pos` (a residue "
    "is assessable for a chaperone contact only if it maps into the 497-kinase "
    "alignment the contacts are transferred through). "
    f"Assessable: {ann['chaperone_contact_resolved'].mean():.1%} of rows.")

print("our 6 A contact positions per kinase (unfolded / folded / CDC37):")
display(pd.DataFrame({
    c: ann[ann[c]].groupby("protein").apply(
        lambda g: pd.to_numeric(g.Position, errors="coerce").nunique())
    for c in ("at_hsp90_unfolded", "at_hsp90_folded", "at_cdc37")
}).dropna(how="all").fillna(0).astype(int))
print(f"\nresidues assessable for a chaperone contact: "
      f"{ann.chaperone_contact_resolved.mean():.1%} of rows")

our 6 A contact positions per kinase (unfolded / folded / CDC37):


,at_hsp90_unfolded,at_hsp90_folded,at_cdc37
protein,,,
araf,11,17,14
braf,11,17,14
craf,11,17,14
egfr,11,17,14
erbb2,11,17,14
ksr1,11,16,14
ksr2,11,16,14
mek1,11,16,14
mek2,11,16,14



residues assessable for a chaperone contact: 37.4% of rows


## Step 7 — quantitative predictors

Position- and substitution-level features used by the Fig 4 univariate and
factor analyses. Two conservation axes are kept deliberately: kinase-paralog
JSD (within-kinome) and phyloP (cross-species) are near-independent (Pearson
r = 0.17) and behave differently — buffering tracks kinase-fold-specific
conservation, not generic evolutionary constraint.

In [8]:
def _merge_positional(df, path, cols, rename=None, pos_col="position"):
    src = pd.read_csv(path, sep="\t")
    src = src.rename(columns={pos_col: "_pos", "Position": "_pos"})
    src["_pos"] = pd.to_numeric(src["_pos"], errors="coerce")
    src = src.dropna(subset=["_pos"])
    src["_pos"] = src["_pos"].astype(int)
    src = src[["protein", "_pos"] + cols].drop_duplicates(["protein", "_pos"])
    if rename:
        src = src.rename(columns=rename)
    out = df.copy()
    out["_pos"] = pd.to_numeric(out["Position"], errors="coerce")
    out = out.merge(src, on=["protein", "_pos"], how="left")
    return out.drop(columns="_pos")

# Conservation is jsd_conservation (Capra-Singh) throughout. A Shannon-entropy
# column on the same Modi-Dunbrack alignment is deliberately not used: it is not
# the statistic the conservation axis claims, and it counted the alignment's
# ANNOTATION pseudo-sequence as a sequence (bug B11).
ann = _merge_positional(ann, PHYLOP, ["phylop_vert"])
ann = _merge_positional(ann, KINASE_JSD, ["jsd_conservation"])
ann = _merge_positional(ann, MOTIFS, ["motif"], rename={"motif": "kinase_motif"})

# ddG is per (protein, position, wt, mut) — substitution-level, not positional.
ddg = pd.read_csv(SPURS_DDG, sep="\t")[
    ["protein", "position", "wt_aa", "mut_aa", "ddG_SPURS"]]
ann["_pos"] = pd.to_numeric(ann["Position"], errors="coerce")
ann = ann.merge(
    ddg.rename(columns={"position": "_pos", "wt_aa": "Wild Type Residue",
                        "mut_aa": "Mutation"}),
    on=["protein", "_pos", "Wild Type Residue", "Mutation"], how="left",
).drop(columns="_pos")

for c in ("phylop_vert", "jsd_conservation",
          "ddG_SPURS", "kinase_motif"):
    if c in ann.columns:
        print(f"  {c:20s} non-null {ann[c].notna().mean():6.1%}")

  phylop_vert          non-null  72.9%
  jsd_conservation     non-null  34.3%
  ddG_SPURS            non-null  85.1%
  kinase_motif         non-null  37.3%


## Step 8 — population and cancer observation

gnomAD and All of Us flags come from the HGVSp-keyed population table, which
matches on **genomic coordinates** `(chr, pos, ref, alt)` rather than on
protein-change strings. That matters: the database annotators' protein-change
strings vary by transcript, and matching on them lost ~7% of BRAF/CRAF hits and
9 of HGFR's AoU stop-gains. GENIE counts come from the joined clinical table,
which uses the same genomic-coordinate approach lifted to GRCh37.

`min_nt_changes` and `gnomad_mappable` are carried through because the
depletion analyses must be restricted to variants reachable by a single
nucleotide change — a population database cannot observe anything else, so
including them would bias the null rate downward.

In [9]:
pop = pd.read_csv(POPULATION_HGVSP, sep="\t", low_memory=False)
pop_keyed = (pop.rename(columns={"position": "_pos", "wt_residue": "Wild Type Residue",
                                 "mutation": "Mutation"})
             [["protein", "_pos", "Wild Type Residue", "Mutation",
               "in_gnomad", "in_aou", "min_nt_changes", "gnomad_mappable",
               "nmd_zone"]]
             .drop_duplicates(["protein", "_pos", "Wild Type Residue", "Mutation"]))
ann["_pos"] = pd.to_numeric(ann["Position"], errors="coerce")
ann = ann.merge(pop_keyed, on=["protein", "_pos", "Wild Type Residue", "Mutation"],
                how="left").drop(columns="_pos")

genie = (pd.read_csv(GENIE_JOINED, sep="\t", low_memory=False)
         [["protein", "variant", "genie_sriram_count"]]
         .rename(columns={"genie_sriram_count": "genie_count"})
         .drop_duplicates(["protein", "variant"]))
ann = ann.merge(genie, on=["protein", "variant"], how="left")

for c in ("in_gnomad", "in_aou", "min_nt_changes", "gnomad_mappable",
          "genie_count"):
    print(f"  {c:18s} non-null {ann[c].notna().mean():6.1%}")
print(f"\nSNV-accessible + mappable missense: "
      f"{int(((ann.min_nt_changes == 1) & (ann.gnomad_mappable == True) & (ann['Mutation Type'] == 'missense')).sum()):,}")

  in_gnomad          non-null  91.0%
  in_aou             non-null  91.0%
  min_nt_changes     non-null  84.7%
  gnomad_mappable    non-null  91.0%
  genie_count        non-null  77.9%

SNV-accessible + mappable missense: 178,217


## Step 9 — reference accessions, and a wild-type sequence audit

Attaches the protein-level reference accessions and then *proves* the numbering,
rather than asserting it. Every distinct `(protein, Position, Wild Type Residue)`
in the table is checked against the actual reference sequence in
`data/reference_protein_sequences.fasta`.

Only **protein** accessions are carried. The libraries are built on recoded
wild-type ORFs, so the construct's nucleotide sequence is not any reference
transcript — a `c.` description against `NM_`/`ENST` would be wrong for every
variant. The matching transcript lives in `config/protein_accessions.yaml` as
provenance, because it is how each protein record was verified.

Each accession was verified by sequence, not by cross-reference:
`scripts/verify_ensembl_proteins.py` and `scripts/verify_refseq_transcripts.py`
require a character-for-character match to the UniProt isoform our positions were
validated against. That test rejected two of UniProt's own Ensembl
cross-references (BRAF, SHP2) and found that KSR1 has no identical Ensembl
translation at all — RefSeq still resolves it, as the predicted model
`XP_011523731.1`.

Three proteins are deliberately **not** MANE Select, and `mane_select` records it:
KRAS (the library is KRAS4B, MANE is KRAS4A), KSR1 (923-aa UniProt canonical vs
MANE's 928-aa form) and BRAF (the classic 766-aa numbering that makes the hotspot
V600 rather than V640).

The audit tolerates exactly one class of exception: a position one past the end of
the protein in a **C-terminally tagged** library. Those ORFs are fused to the MCP
tag and so have no stop codon of their own, and the residue there belongs to the
linker. ERBB2 `T1256fs` is the only such row.

In [10]:
import yaml

ACCESSIONS = PROJECT_ROOT / "config" / "protein_accessions.yaml"
REF_FASTA = PROJECT_ROOT / "data" / "reference_protein_sequences.fasta"

ACC = yaml.safe_load(ACCESSIONS.read_text())
REF = {}
for _blk in REF_FASTA.read_text().split(">")[1:]:
    _ln = _blk.strip().split("\n")
    REF[_ln[0].split()[0]] = "".join(_ln[1:])
print(f"reference sequences: {len(REF)} proteins, "
      f"{sum(map(len, REF.values())):,} residues")

# ---- attach the accessions (idempotent: overwrite rather than merge-duplicate)
_p = ann["library"].str.replace(r"_(nterm|cterm)$", "", regex=True)
for _col, _key in (("uniprot_id", "uniprot"), ("ensembl_protein", "ensembl_protein"),
                   ("refseq_protein", "refseq_protein"), ("mane_select", "mane_select")):
    ann[_col] = _p.map(lambda q: (ACC.get(q) or {}).get(_key))
print("\naccessions attached:")
print(ann.groupby(_p)[["uniprot_id", "ensembl_protein", "refseq_protein",
                       "mane_select"]].first().to_string())

# ---- audit every wild-type residue against the reference sequence
CTERM_TAGGED = {q for q, v in ACC.items()
                if (v or {}).get("mcp_tag_position", "") == "C-terminal"} or {
    "met", "ret", "egfr", "erbb2", "sos1", "sos2"}
_chk = ann.loc[~ann["Mutation Type"].isin(["standard", "wild type"]),
               ["library", "variant", "Position", "Wild Type Residue",
                "Mutation Type"]].copy()
_chk["protein"] = _chk["library"].str.replace(r"_(nterm|cterm)$", "", regex=True)
_chk["pos"] = pd.to_numeric(_chk["Position"], errors="coerce")
_chk = _chk.dropna(subset=["pos"])
_chk["pos"] = _chk["pos"].astype(int)
_uniq = (_chk.drop_duplicates(["protein", "pos", "Wild Type Residue"])
         .rename(columns={"Wild Type Residue": "wt"}))

_bad = []
for _r in _uniq.itertuples():
    _s = REF.get(_r.protein)
    if _s is None:
        continue
    _w = _r.wt
    if 1 <= _r.pos <= len(_s):
        if _s[_r.pos - 1] != _w:
            _bad.append((_r.protein, _r.library, _r.variant, _r.pos, _w,
                         _s[_r.pos - 1], "residue differs from reference"))
    elif _r.pos == len(_s) + 1:
        # the stop codon: '*' in an N-terminally tagged library; in a
        # C-terminally tagged one the ORF reads through into the MCP linker,
        # so a real residue here is expected rather than wrong
        if _w != "*" and _r.protein not in CTERM_TAGGED:
            _bad.append((_r.protein, _r.library, _r.variant, _r.pos, _w, "(stop)",
                         "stop-codon position, WT is not '*'"))
    else:
        _bad.append((_r.protein, _r.library, _r.variant, _r.pos, _w, "(beyond)",
                     "position past the end of the protein"))

print(f"\nwild-type audit: {len(_uniq):,} distinct (protein, position, residue) "
      f"triples across {_uniq['protein'].nunique()} proteins")
if _bad:
    _b = pd.DataFrame(_bad, columns=["protein", "library", "variant", "position",
                                     "our_wt", "reference", "reason"])
    print(f"  DISAGREEMENTS: {len(_b)}")
    print(_b.to_string(index=False))
else:
    print("  100% agreement at the amino-acid level")

_tag_junction = _uniq[(_uniq["protein"].isin(CTERM_TAGGED))
                      & (_uniq["pos"] == _uniq["protein"].map(
                          lambda q: len(REF.get(q, "")) + 1))]
if len(_tag_junction):
    print(f"\n  tolerated -- MCP-linker junction in a C-terminally tagged library "
          f"({len(_tag_junction)} position(s)):")
    print(_tag_junction[["library", "variant", "pos", "wt"]]
          .to_string(index=False))

assert not _bad, (
    f"{len(_bad)} wild-type residue(s) disagree with the reference sequence — the "
    "position numbering or the accession is wrong; see the table above")
provenance.append(
    f"Reference accessions attached (uniprot_id / ensembl_protein / refseq_protein "
    f"/ mane_select) from config/protein_accessions.yaml, each verified by "
    f"sequence rather than by cross-reference. All {len(_uniq):,} distinct "
    f"(protein, position, wild-type residue) triples agree with the reference "
    f"sequence; the only tolerated exception is the MCP-linker junction one past "
    f"the end of a C-terminally tagged ORF (ERBB2 T1256fs).")

reference sequences: 17 proteins, 13,526 residues



accessions attached:
        uniprot_id    ensembl_protein  refseq_protein  mane_select
library                                                           
araf        P10398  ENSP00000366244.4     NP_001645.1         True
braf        P15056  ENSP00000815007.1     NP_004324.2        False
craf        P04049  ENSP00000251849.4     NP_002871.1         True
egfr        P00533  ENSP00000275493.2     NP_005219.2         True
erbb2       P04626  ENSP00000269571.4     NP_004439.2         True
grb2        P62993  ENSP00000339007.4     NP_002077.1         True
kras      P01116-2  ENSP00000308495.3     NP_004976.2        False
ksr1        Q8IVT5                NaN  XP_011523731.1        False
ksr2        Q6VAB6  ENSP00000339952.4     NP_775869.4         True
mek1        Q02750  ENSP00000302486.5     NP_002746.1         True
mek2        P36507  ENSP00000262948.4     NP_109587.1         True
met         P08581  ENSP00000380860.3     NP_000236.2         True
mras        O14807  ENSP00000389682.2  N


wild-type audit: 9,981 distinct (protein, position, residue) triples across 17 proteins
  100% agreement at the amino-acid level


## Step 10 — write the combined table and the provenance report

The report is the audit trail: every place where the delivered file was
modified or where a decision was taken that a reader might otherwise have to
reverse-engineer.

In [11]:
assert ann.shape[0] == scores.shape[0], (
    f"row count changed during annotation: {scores.shape[0]:,} -> {ann.shape[0]:,} "
    "— a merge introduced duplicate keys")

ANNOTATED_COMBINED.parent.mkdir(parents=True, exist_ok=True)
ann.to_csv(ANNOTATED_COMBINED, sep="\t", index=False)
print(f"wrote {ANNOTATED_COMBINED.relative_to(utils.PROJECT_ROOT)}")
print(f"  {ann.shape[0]:,} rows x {ann.shape[1]} columns\n")

print("=" * 78)
print("PROVENANCE — modifications and decisions applied to the delivered file")
print("=" * 78)
for i, line in enumerate(provenance, 1):
    print(f"\n{i}. {line}")

print("\n" + "=" * 78)
print("OPEN ISSUES (measured against THIS pipeline, 2026-08-14)")
print("=" * 78)
# Every entry re-measured on this pipeline's own output. Two of them are bugs in
# our code rather than anything upstream.
for line in [
    "OURS: chap_dep_classification labels are inverted relative to the quantity "
    "they classify -- 'decreased' rows have a median Abund_chaperone_dependency "
    "of +0.32 and 'increased' rows -0.34. Both columns are dropped in Step 3 and "
    "the buffering classes are recomputed in HSP90i.ipynb, so nothing downstream "
    "reads them, but add_chaperone_dependency should be fixed.",

    "KRAS shows 10.5% pdb_aa_mismatch, the only protein above 0. The AlphaFold "
    "model is P01116 canonical (KRAS4A) while the library is KRAS4B, so the two "
    "disagree over the C-terminal hypervariable region. plddt and pdb_aa for "
    "those KRAS positions describe the wrong isoform and should not be used.",
    "column is usable until that is resolved.",

    "BRAF abundance x CIAR (16,701 rows) still has no analysis.",

    "The DN ontology documents its active-site category in terms of a manual "
    "annotation column this pipeline does not build, so "
    "docs/dn_ontology.md should be reworded around `active_site` as the "
    "annotation engine computes it from the NCBI CD features.",
]:
    print(f"  - {line}")


wrote output/annotated_combined.tsv
  531,797 rows x 114 columns

PROVENANCE — modifications and decisions applied to the delivered file

1. Base layer is scores_reannotated.tsv (531,797 rows): our scores, built from our own barcode counts by Scoring.ipynb, with every annotation rebuilt from primary sources by scripts/reannotate_scores.py and scripts/compute_structure_annotations.py.

2. Interfaces: one column, `interface_pdb_curated`, from the augmented curated interactions file, which the annotation engine reads directly. Chaperone contacts are computed from 7ZR0 and 8U1L in Step 6.

3. DN thresholds recomputed from control_barcode_scores.tsv rather than read from the delivered dn_cutoffs_empty_vector.tsv: 2.5th percentile of 200 draws of a 10-barcode mean, median over 200 seed repeats. 22 baseline activity cells carry a threshold.

4. Interfaces: 1,433 unique positions across 17 proteins, from the augmented curated file, with the contacting partner recorded per position. The augment